# V1 — Vector Diagnostics: structure & semantic similarity for ANY representation

A reusable battery. Point it at any hidden representation `X` (shape
`[n, d]`) — LLM pooled states, image embeddings, sentence vectors — and it
runs every structural test from the project. If you also have a **paired**
representation `Y` (row i = same input in another model), it runs the
cross-space battery too: CKA, Procrustes-vs-ridge, volume knobs, retrieval,
membership gap.

| Block | Tests | Needs |
|---|---|---|
| 1. Sanity | finite / zero rows / duplicates / norms / rows-per-dim | X |
| 2. Anisotropy + hubness | mean pairwise cosine, mean-direction ratio, PC1 & top-k share, rank-1 energy, effective rank, spectrum plot | X |
| 3. Robustness | centering, top-PC removal, **whitening** — before/after | X |
| 4. Cross-space | held-out ridge R² + shuffle test, Procrustes vs ridge, CKA, singular values of W (volume knobs) | X, Y paired |
| 5. Semantic | R@K retrieval, membership gap (true-twin minus best impostor) | X, Y paired |

Every block prints calibrated reference points next to the measured value.

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr

DATA_DIR = Path(os.environ["DATA_DIR"])
rng = np.random.default_rng(0)

# ---------- POINT THESE AT ANY REPRESENTATION ----------
X_FILE, X_KEY = "pairs.npz", "mob_img"      # required: [n, d1]
Y_FILE, Y_KEY = "pairs.npz", "sig_img"      # optional paired [n, d2]; None to skip
# Y_FILE = None

X = np.load(str(DATA_DIR / X_FILE))[X_KEY].astype(np.float64)
Y = (np.load(str(DATA_DIR / Y_FILE))[Y_KEY].astype(np.float64)
     if Y_FILE else None)
print("X", X.shape, "| Y", None if Y is None else Y.shape)

def l2n(V):
    return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)

## Block 1 — Sanity (run before believing anything)

In [ ]:
n, d = X.shape
print(f"rows/dim: {n}/{d} = {n/d:.1f}   (fitted maps need >=5)")
bad = int(np.isnan(X).any(1).sum() + np.isinf(X).any(1).sum())
norms = np.linalg.norm(X, axis=1)
nz = int((norms < 1e-9).sum())
samp = X[rng.choice(n, min(4000, n), replace=False)]
_, cnt = np.unique(np.round(samp, 5), axis=0, return_counts=True)
dup = int((cnt > 1).sum())
print(f"NaN/Inf rows {bad} | zero rows {nz} | duplicate rows {dup}")
print(f"norms: mean {norms.mean():.3f}  min {norms.min():.3f}  "
      f"max {norms.max():.3f}"
      + ("   <- unit-normalized" if abs(norms.mean()-1) < 1e-3 else ""))

## Block 2 — Anisotropy battery

| Statistic | isotropic looks like | concentrated looks like |
|---|---|---|
| mean random-pair cosine | ~0.00 | >0.3 |
| mean-direction ratio | ~0.0 | >0.5 |
| PC1 share | ~1/d | >0.3 |
| rank-1 energy | small | >0.5 |
| effective rank | ~d | << d |

In [ ]:
def anisotropy_report(M, name):
    Mn = l2n(M)
    k = min(2000, len(M))
    ii = rng.choice(len(M), k); jj = rng.choice(len(M), k)
    keep = ii != jj
    pair_cos = float((Mn[ii[keep]] * Mn[jj[keep]]).sum(1).mean())
    mdr = float(np.linalg.norm(M.mean(0)) /
                max(np.linalg.norm(M, axis=1).mean(), 1e-12))
    Xc = M - M.mean(0, keepdims=True)
    s = np.linalg.svd(Xc, full_matrices=False, compute_uv=False)
    e = s ** 2
    p = e / e.sum()
    pc1 = float(p[0]); topk = float(p[:10].sum())
    rank1 = pc1
    eff_rank = float(np.exp(-(p[p > 0] * np.log(p[p > 0])).sum()))
    pr = float(e.sum() ** 2 / (e ** 2).sum())     # participation ratio
    # hubness: a DIFFERENT pathology from anisotropy. Anisotropy is
    # about the variance spectrum; hubness is about the neighbourhood
    # graph - a few items becoming everyone's nearest neighbour. A
    # hubbed space can pass verification while failing retrieval.
    from scipy.stats import skew as _skew
    sub = Mn[rng.permutation(len(Mn))[:min(2000, len(Mn))]]
    Sm = sub @ sub.T
    np.fill_diagonal(Sm, -9)
    nn = np.argsort(-Sm, 1)[:, :10]
    hub = float(_skew(np.bincount(nn.ravel(), minlength=len(sub))))
    print(f"--- {name}  [n={M.shape[0]}, d={M.shape[1]}]")
    print(f"  mean random-pair cosine : {pair_cos:+.3f}   (isotropic ~0)")
    print(f"  mean-direction ratio    : {mdr:.3f}    (isotropic ~0)")
    print(f"  PC1 / top-10 var share  : {pc1:.3f} / {topk:.3f}")
    print(f"  rank-1 energy share     : {rank1:.3f}")
    print(f"  hubness (N10 skewness)  : {hub:+.2f}   "
          f"(healthy <~2; >3 is hubbed)")
    print(f"  effective rank (entropy): {eff_rank:.1f} of {M.shape[1]}")
    print(f"  participation ratio     : {pr:.1f}")
    return s

sX = anisotropy_report(X, "X")
if Y is not None:
    sY = anisotropy_report(Y, "Y")

plt.figure(figsize=(7.5, 4))
plt.semilogy(sX ** 2 / (sX ** 2).sum(), label="X")
if Y is not None:
    plt.semilogy(sY ** 2 / (sY ** 2).sum(), label="Y")
plt.xlabel("principal direction"); plt.ylabel("variance share (log)")
plt.title("Spectrum: flat = isotropic, steep = concentrated")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(str(DATA_DIR / "v1_spectrum.png"), dpi=140); plt.show()

## Block 3 — Does structure survive the corrections?

Center, then remove top principal components. A finding that survives is
not an artifact of a shared dominant direction; one that vanishes was.

In [ ]:
def remove_top_pcs(M, k):
    Mc = M - M.mean(0, keepdims=True)
    U, s, Vt = np.linalg.svd(Mc, full_matrices=False)
    return Mc - (Mc @ Vt[:k].T) @ Vt[:k]

for k in (0, 1, 3, 10):
    Xk = remove_top_pcs(X, k) if k else X - X.mean(0, keepdims=True)
    Xn = l2n(Xk)
    kk = min(2000, len(X))
    ii = rng.choice(len(X), kk); jj = rng.choice(len(X), kk)
    keep = ii != jj
    pc = float((Xn[ii[keep]] * Xn[jj[keep]]).sum(1).mean())
    print(f"  top-{k} PCs removed: mean random-pair cosine {pc:+.3f}")
print()
print("HOW TO READ THIS. k=0 is centering alone (subtract the mean")
print("vector); k>=1 also deletes that many principal directions.")
print("  - big drop already at k=0  -> a shared MEAN DIRECTION carried")
print("    the similarity. Centering fixes the symptom, and the space")
print("    may already be reasonably isotropic underneath.")
print("  - little change at k=0, big drop at k=1/3 -> a few dominant")
print("    PRINCIPAL directions carried it, not a common offset.")
print("  - pair-cosine near 0 at every k while effective rank stays")
print("    LOW -> the variance is concentrated, not merely offset.")
print("    That is the GPT-2 signature, and only rescaling (whitening)")
print("    helps: centering drove pair-cos +0.999 -> -0.019 there while")
print("    effective rank stayed at 6.5 and retrieval got WORSE.")
print("Always read this block together with effective rank above:")
print("pair-cosine alone is not a sufficient diagnostic.")


# ---- whitening: the correction that mattered most (Section C.11) ----
# Top-PC removal deletes a few directions. Whitening RESCALES every
# direction to equal variance, which is a strictly stronger correction
# and the one that recovered R@1 0.130 -> 0.478 in this project.
# NOTE the discipline: in a real evaluation the covariance must be
# estimated on TRAIN rows only and applied unchanged to held-out rows.
def whiten_fit(Xtr, eps=1e-6):
    mu = Xtr.mean(0)
    C = np.cov((Xtr - mu).T) + eps * np.eye(Xtr.shape[1])
    ev, V = np.linalg.eigh(C)
    ev = np.clip(ev, 1e-12, None)
    Wm = V @ np.diag(ev ** -0.5) @ V.T
    return lambda A: (A - mu) @ Wm

_wh = whiten_fit(X)                      # in-sample here: diagnostic only
Xw = _wh(X)
_Xn = l2n(Xw)
_kk = min(2000, len(X))
_ii = rng.choice(len(X), _kk); _jj = rng.choice(len(X), _kk)
_keep = _ii != _jj
_pc = float((_Xn[_ii[_keep]] * _Xn[_jj[_keep]]).sum(1).mean())
_Xc = Xw - Xw.mean(0)
_s = np.linalg.svd(_Xc, full_matrices=False, compute_uv=False)
_p = _s ** 2 / (_s ** 2).sum(); _p = _p[_p > 0]
_er = float(np.exp(-(_p * np.log(_p)).sum()))
print(f"\n  WHITENED: mean random-pair cosine {_pc:+.3f}, "
      f"effective rank {_er:.1f} of {X.shape[1]}")
print("  compare with the raw effective rank in Block 2. A large rise")
print("  here means the space was readable all along and the coordinate")
print("  frame was hiding it - refit the cross-space battery against a")
print("  whitened target before reporting any negative result.")
print("\n  CAUTION (Section C.12): whitening helps when READING a known")
print("  correspondence and HURTS when DISCOVERING an unknown one - it")
print("  collapsed unsupervised matching from 39.7% to 0.1%. Do not")
print("  whiten before a shape-matching / Gromov-Wasserstein step.")

## Block 4 — Cross-space battery (needs paired Y)

Ridge vs Procrustes on the same split answers *which kind* of difference
separates the spaces; the fitted W's singular values are its volume knobs.

In [ ]:
if Y is not None:
    idx = rng.permutation(len(X))
    ktr = int(0.75 * len(X))
    tr, te = idx[:ktr], idx[ktr:]

    # ridge, held-out
    a = 1e-2
    Wm = np.linalg.solve(X[tr].T @ X[tr] + a * np.eye(X.shape[1]),
                         X[tr].T @ Y[tr])
    P = X[te] @ Wm
    r2 = 1 - ((Y[te] - P) ** 2).sum() / ((Y[te] - Y[te].mean(0)) ** 2).sum()
    cos_w = float((l2n(P) * l2n(Y[te])).sum(1).mean())

    # shuffle control (alignment + power check)
    Ys = Y[tr][rng.permutation(ktr)]
    Ws = np.linalg.solve(X[tr].T @ X[tr] + a * np.eye(X.shape[1]),
                         X[tr].T @ Ys)
    r2s = 1 - ((Y[te] - X[te] @ Ws) ** 2).sum() /               ((Y[te] - Y[te].mean(0)) ** 2).sum()

    # procrustes (zero-pad if widths differ)
    d1, d2 = X.shape[1], Y.shape[1]
    Xp = np.pad(X, ((0, 0), (0, max(0, d2 - d1))))[:, :max(d1, d2)]
    Yp = np.pad(Y, ((0, 0), (0, max(0, d1 - d2))))[:, :max(d1, d2)]
    U, _, Vt = np.linalg.svd(Xp[tr].T @ Yp[tr])
    R = U @ Vt
    Pr = Xp[te] @ R
    r2p = 1 - ((Yp[te] - Pr) ** 2).sum() /               ((Yp[te] - Yp[te].mean(0)) ** 2).sum()
    cos_p = float((l2n(Pr) * l2n(Yp[te])).sum(1).mean())

    # CKA on the (sub)samples
    def cka(Xa, Ya):
        Xa = Xa - Xa.mean(0); Ya = Ya - Ya.mean(0)
        num = np.linalg.norm(Ya.T @ Xa) ** 2
        return float(num / (np.linalg.norm(Xa.T @ Xa) *
                            np.linalg.norm(Ya.T @ Ya)))
    sub = rng.choice(len(X), min(4000, len(X)), replace=False)
    print(f"held-out ridge   : R2 {r2:.3f}   cosine {cos_w:.3f}")
    print(f"shuffle control  : R2 {r2s:.3f}   "
          f"(honest fit must beat this by >0.2)")
    print(f"procrustes       : R2 {r2p:.3f}   cosine {cos_p:.3f}")
    print(f"  ridge >> procrustes on R2 while procrustes cosine is high")
    print(f"  -> anisotropy mismatch (volumes), not geography")
    print(f"CKA(X, Y)        : {cka(X[sub], Y[sub]):.3f}")

    sw = np.linalg.svd(Wm, compute_uv=False)
    print(f"\nvolume knobs (singular values of W):")
    print(f"  max {sw.max():.3f}  median {np.median(sw):.3f}  "
          f"min {sw.min():.3f}  spread {sw.max()/max(sw.min(),1e-9):.0f}x")
    print(f"  fraction outside [0.8, 1.25]: "
          f"{((sw < 0.8) | (sw > 1.25)).mean():.2f}")
    print("  a pure rotation would print all ~1.0")
else:
    print("no Y - skipping cross-space battery")

## Block 5 — Semantic similarity: retrieval and membership (needs paired Y)

In [ ]:
if Y is not None:
    q = l2n(X[te] @ Wm)          # adapted queries
    g = l2n(Y[te])               # gallery
    S = q @ g.T
    order = np.argsort(-S, axis=1)
    ranks = (order == np.arange(len(S))[:, None]).argmax(1)
    for k in (1, 5, 10):
        print(f"  R@{k} = {(ranks < k).mean():.3f}")
    tc = S[np.arange(len(S)), np.arange(len(S))]
    Sw = S.copy(); Sw[np.arange(len(S)), np.arange(len(S))] = -1
    gap = tc - Sw.max(1)
    print(f"  membership gap: mean {gap.mean():+.3f}  min {gap.min():+.3f}"
          f"  positive {(gap > 0).mean():.3f}")
    print("  large positive gap -> a threshold answers 'is this item")
    print("  already indexed?' across the two spaces")

    # verification AUC: a WEAKER question than R@1 - separate a true
    # pair from a TYPICAL random one, rather than outranking every
    # candidate. Reporting both localises a failure (Appendix D.2):
    # AUC high with R@1 low = coarse correspondence kept, pair-level
    # resolution lost.
    _j = rng.permutation(len(te))
    _j = np.where(_j == np.arange(len(te)), (_j + 1) % len(te), _j)
    _pos = (q * g).sum(1)
    _neg = (q * g[_j]).sum(1)
    _y = np.r_[np.ones(len(_pos)), np.zeros(len(_neg))]
    _sc = np.r_[_pos, _neg]
    _o = np.argsort(-_sc); _y = _y[_o]
    _tpr = np.cumsum(_y) / _y.sum()
    _fpr = np.cumsum(1 - _y) / (1 - _y).sum()
    _auc = float(np.trapezoid(_tpr, _fpr))
    print(f"  verification AUC: {_auc:.4f}  "
          f"(positives mean {_pos.mean():+.3f}, "
          f"negatives {_neg.mean():+.3f})")
    print("  AUC measures SEPARATION, not magnitude: small absolute")
    print("  cosines with a clean split still give AUC near 1.")
else:
    print("no Y - skipping semantic battery")

## Reading the whole battery

- **Structure** (blocks 2-3) tells you the *shape* of one space: how many
  directions carry variance, whether similarity scores are inflated by a
  shared component, whether any finding survives centering.
- **Semantics** (blocks 4-5) needs a second, paired space: it tells you
  whether the two spaces carry the *same content* (CKA, ridge), what kind
  of transformation separates them (Procrustes-vs-ridge; the volume
  knobs), and whether the correspondence is good enough to *use*
  (R@K, membership gap).
- Any negative result here inherits the project's rule: verify rows/dim,
  pass the shuffle test, and check artifact sanity before believing it.